# GenAI-Kickstart
## Lesson: Basic RAG

This lesson is intended to show you the basics of a Retrieval Augmented Generation (RAG) system.

During this lesson you will learn how to ...

- implement the different building blocks of RAG
- create a ingestion pipeline from the building blocks
- create a retrieval pipeline from the building blocks
- use a RAG system to generate responses to user inputs

### Set up the environment

In [1]:
import sys
sys.path.append('../../../')

In [2]:
from dotenv import load_dotenv, find_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb import EphemeralClient
import uuid
from lib.completion import generate_completion
from lib.constants import PROCESSED_DATA_PATH
from lib.validation import validate_environment_for_model
from litellm import embedding
import os

load_dotenv(find_dotenv())


True

In [3]:
# Set values for model, model parameters and prompt

# Models
GENERATION_PROVIDER = os.getenv("DEFAULT_GENERATION_PROVIDER")
GENERATION_MODEL = os.getenv("DEFAULT_GENERATION_MODEL")
EMBEDDING_PROVIDER =  os.getenv("DEFAULT_EMBEDDING_PROVIDER")
EMBEDDING_MODEL = os.getenv("DEFAULT_EMBEDDING_MODEL") # Check you .env file, if the respective embedding model is available. If not, update the .env file and restart the kernel!

# Generation parameters
TEMPERATURE = 0.9 
MAX_OUTPUT_TOKENS = 600

# Retrieval parameters
TOP_K = 3
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

# Prompts
SYSTEM_PROMPT = "Your are a friendly assistant"

In [4]:
# Set the full model names
generation_model = f"{GENERATION_PROVIDER}/{GENERATION_MODEL}"
embedding_model = f"{EMBEDDING_PROVIDER}/{EMBEDDING_MODEL}"

# Check if all envirnment variables are set
validate_environment_for_model(model=generation_model)
validate_environment_for_model(model=embedding_model)

## Generate response without internal knowledge


In [5]:
# Let's try asking a specific question about a Sherlock Holmes book.
user_prompt = "Lucy noticed a number on the ceiling when taking breakfast. Which number was written into the ceiling?" # The answer should contain the number "28"

generate_completion(
    user_prompt=user_prompt,
    model_name=generation_model,
)

'This is a classic riddle!\n\nThe number was a **9**.\n\nWhen you look up at a "9" written on the ceiling from below, it appears to be a "6"!'

In [6]:
# Let's give the model a little bit more context
user_prompt = "In one Sherlock Holmes book, Lucy noticed a number on the ceiling when taking breakfast. Which number was written into the ceiling?" # The answer should contain the number "28"

generate_completion(
    user_prompt=user_prompt,
    model_name=generation_model,
)

"That's an interesting detail, but it actually sounds like you might be thinking of a different book!\n\nIn **Anne Frank's diary**, Anne Frank mentions seeing the number **"

## Define helpers

In [7]:
# This will be the chromadb collection we use as a knowledge base. We do not need the in-memory EphemeralClient.
chromadb_collection = EphemeralClient().get_or_create_collection(name="default")

# Have a look into the knowledgebase
def peek_knowledgebase():
  """Shows the first ten items of the knowledgebase"""
  print(chromadb_collection.peek())

### Define RAG Building Blocks

In [8]:
# Get content of books. The content will already be cleansed.
def load_file_content(file_name: str) -> str:
  """ Loads content of a file in the local file systemby a given file name and return its content

    Parameters
    ----------
    file_name : str
        The name of the file to be loaded
    Returns
    -------
    file_content : str
        the content of the file as a string
  """
  with open(f"{PROCESSED_DATA_PATH / file_name}", "r") as f:
    return f.read()

In [9]:
# Building Block "Chunking": Split the content into smaller chunks
def do_chunk(text: str) -> list[str]:
  """ Chunks a given text by a given chunk size and chunk overlap and returns a list of chunks

  Parameters
  ----------
  text : str
      The text to be chunked
  chunk_size : int, optional [default: DEFAULT_CHUNK_SIZE]
        The desired chunk size
  chunk_overlap : int, optional [default: DEFAULT_CHUNK_OVERLAP]
        The desired chunk overlap
  Returns
  -------
  chunks: [str]
      The created chunks
  """
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=CHUNK_SIZE,
      chunk_overlap=CHUNK_OVERLAP,
      length_function=len,
  )
  return text_splitter.split_text(text=text)

In [10]:
# Building Block "Embedding": Create multi dimensional embeddings for a given chunk.
def do_embed(chunk: str) -> list[float]:
  """ Embeds a given chunk and returns the embedding

  Parameters
  ----------
  chunk : str
      The chunk to be embedded
  Returns
  -------
  embedding: [float]
      The created embedding
  """
  response = embedding(
          model=embedding_model,
          input=chunk
      )
  return response["data"][0]["embedding"]


def do_batch_embed(chunks: list[str], batch_size: int = 100) -> list[list[float]]:
  """ Embeds a list of chunks and returns the embeddings

  Parameters
  ----------
  chunks : [str]
      The chunks to be embedded
  Returns
  -------
  embeddings: [list[float]]
      The created embeddings
  """

  all_embeddings = []

  # Process chunks in batches
  for i in range(0, len(chunks), batch_size):
      batch = chunks[i:i + batch_size]
      response = embedding(
          model=embedding_model,
          input=batch
      )
      batch_embeddings = [data["embedding"] for data in response["data"]]
      all_embeddings.extend(batch_embeddings)

  return all_embeddings

In [11]:
# Building Block "Knowledgebase": Store embeddings and the corresponding content in a vectorstore
def persist_embeddings(chunks: list[str], embeddings: list[float])-> None:
  """ Persists the embeddings and the chunks in the knowledgebase

  Parameters
  ----------
  chunks : [str]
      The chunks to be persisted
  embeddings: [list[float]]
      The corresponding embeddings to be persisted
  """
  ids = [str(uuid.uuid4()) for _ in chunks]
  chromadb_collection.add(ids=ids, documents=chunks, embeddings=embeddings)

In [12]:
# Building Block "Augmentation": Create an updated prompt by merging the original user input with the provided context
def augment(user_input: str, context: list[str]) -> str:
  """ Augments a given user input by merging it with the provided context and returns the augmented prompt

  Parameters
  ----------
  user_input : str
      The user input to be augmented
  context : [str]
      The context to be merged with the user input
  Returns
  -------
  augmented_prompt: str
      The created augmented prompt
  """
  prepared_context = "\n".join(context)
  augmented_prompt = f"""
    Answer the question as detailed as possible from the provided context, make sure to provide all the details, if the answer is not in
    provided context just say, "answer is not available in the context", don't provide the wrong answer\n\n
    Context:\n{prepared_context}\n
    Question: \n{user_input}\n

    Answer:
  """
  return augmented_prompt

In [13]:
# Building Block "Top-k Fetching": Get the k semantically closest chunks to the user input from the knowledgebase
def do_top_k_fetching(user_input_embedding: list[float], top_k: int) -> list[str]:
  """ Fetches the k semantically closest chunks to the user input from the knowledgebase

  Parameters
  ----------
  user_input_embedding : [float]
      The embedding of the user input
  top_k : int
      The number of semantically closest chunks to be fetched

  Returns
  -------
  context: [str]
      The fetched chunks
  """
  # Since we will do the fetching always only for one user_input,
  # instead of querying for multiple embeddings simultanously as allowed by the choma API,
  # we add the embeddings below to a list and return only the first document (chunk)
  return chromadb_collection.query(
      query_embeddings=[user_input_embedding],
      n_results=top_k,
  )["documents"][0]

In [14]:
# Building Block "Generation": Use the generation model to create a response
def generate_response(prompt: str) -> None:
  """ Generates a response for a given prompt

  Parameters
  ----------
  prompt : str
      The prompt to be used for the generation
  """
  response = generate_completion(
    model_name=generation_model,
    user_prompt=prompt,
  )
  print(response)

### Create the ingestion pipeline

In [15]:
def do_ingestion(file_names: list[str]) -> None:
  """ Ingests a list of files by a given file name

  Parameters
  ----------
  file_names : [str]
      The names of the files to be ingested
  """
  # Ingest file by file
  for file_name in file_names:
    # Load prepared book content
    file_content = load_file_content(file_name)

    # Chunk the content into smaller chunks
    chunks = do_chunk(file_content)

    # Embed the chunks
    embeddings = do_batch_embed(chunks)

    # Persist the embeddings and the chunks in the knowledgebase
    persist_embeddings(chunks, embeddings)


### Perform ingestion

In [16]:
# Define file names to be ingested
file_names = ['study_in_scarlett.txt']

# Perform ingestion. Depending on the chunk_size this might take some minutes.
do_ingestion(file_names)

In [17]:
# Use helper function to peek into knowledgebase
peek_knowledgebase()

{'ids': ['23eb035e-a06f-494a-9f99-4568996dadf8', 'e67b4387-e622-4278-8015-4843f7e23f14', '3f615711-d5f4-45aa-9375-8c1537dbf9d7', '9cb37205-df32-4386-b1fc-68664bb7650b', '2648bc53-7860-4af5-8af9-f2dffb2ba03d', '69b4b3b8-aa5c-454c-a78d-f02fd719a487', '85b1c39b-1cfd-4942-8236-986a342832ad', 'a13276b7-2830-4743-ba9e-25cef5920034', 'e15e05f3-e823-41e6-9181-4e02380f570d', '06801959-6467-4319-9d9e-87786e203bf1'], 'embeddings': array([[ 0.04751711, -0.01565197, -0.00866862, ..., -0.00320176,
        -0.01370482, -0.00620819],
       [ 0.0424248 , -0.08199257, -0.0115255 , ..., -0.00353395,
         0.00013127, -0.05247008],
       [-0.00039717,  0.02787378, -0.00904845, ..., -0.02170323,
         0.01917763, -0.02261096],
       ...,
       [ 0.01257239, -0.04925754, -0.00737785, ...,  0.00614621,
         0.01633749,  0.00981242],
       [ 0.03384783, -0.02790589, -0.00635046, ..., -0.0059059 ,
         0.03353002,  0.00262967],
       [ 0.00642045, -0.04901566, -0.00997372, ...,  0.02728618,

### Exercise 01: Create RAG pipeline
In this exercise you will create a rag pipeline for retrieving relevant chunks and generating a grounded response

In [18]:
def do_rag(user_input: str, verbose: bool = False) -> None:
  """ Runs the RAG pipeline with a given user input and prints the response

  Parameters
  ----------
  user_input : str
      The user input to be used for the RAG pipeline
  verbose : bool, optional [default: False]
      Whether to print details of the RAG process or not. Defaults to False
  """
  # TODO: Embed the user input
  user_input_embedding = do_embed(chunk=user_input)

  # TODO: "R" like "Retrieval": Get the k semantically closest chunks to the user input from the knowledgebase
  context = do_top_k_fetching(user_input_embedding=user_input_embedding, top_k=TOP_K)
  if verbose:
    print(f'Retrieved context:\n {context}')

  # TODO: "A" like "Augmented": Create the augmented prompt
  augmented_prompt = augment(user_input=user_input, context=context)
  if verbose:
    print(f'Augmented prompt:\n {augmented_prompt}')

  # TODO: "G" like "Generation": Generate a response
  generate_response(prompt=augmented_prompt)


In [19]:
# Define user input. This should be a question regarding the ingested book
user_input = "Lucy noticed a number on the ceiling when taking breakfast. Which number was written into the ceiling?" # The answer should contain the number "28"

# TODO: Perform rag
do_rag(user_input=user_input, verbose=True)

Retrieved context:
 ['fastened that pin might have struck him to the heart, and he could never have known who had slain him.  Still more shaken was he next morning. They had sat down to their breakfast when Lucy with a cry of surprise pointed upwards. In the centre of the ceiling was scrawled, with a burned stick apparently, the number 28. To his daughter it was unintelligible, and he did not enlighten her. That night he sat up with his gun and kept watch and ward. He saw and he heard nothing, and yet in the morning a great 27 had been painted upon the outside of his door.  Thus day followed day; and as sure as morning came he found that his unseen enemies had kept their register, and had marked up in some conspicuous position how many days were still left to him out of the month of grace. Sometimes the fatal numbers appeared upon the walls, sometimes upon the floors, occasionally they were on small placards stuck upon the garden gate or the railings. With all his vigilance John Ferrie